In [3]:
import requests
import pandas as pd

def fetch_binance_futures_klines(symbol="BTCUSDT", interval="1m", limit=1000):
    """
    從幣安 U 本位合約 API 獲取 K 線數據
    """
    # U本位合約的 API 端點
    url = "https://fapi.binance.com/fapi/v1/klines"
    
    # API 請求參數
    params = {
        "symbol": symbol,
        "interval": interval,
        "limit": limit  # 單次請求最大值為 1500
    }

    try:
        print(f"正在獲取 {symbol} {interval} K線數據...")
        response = requests.get(url, params=params)
        response.raise_for_status()  # 檢查請求是否成功
        data = response.json()
        
        # 根據幣安 API 文件定義的欄位名稱
        columns = [
            "open_time", "open", "high", "low", "close", "volume",
            "close_time", "quote_asset_volume", "number_of_trades",
            "taker_buy_base_asset_volume", "taker_buy_quote_asset_volume", "ignore"
        ]
        
        # 將 JSON 數據轉換為 pandas DataFrame
        df = pd.DataFrame(data, columns=columns)
        
        # 轉換時間戳記 (毫秒) 為可讀的 datetime 格式
        df["open_time"] = pd.to_datetime(df["open_time"], unit="ms")
        df["close_time"] = pd.to_datetime(df["close_time"], unit="ms")
        
        # 將價格與成交量等欄位從字串轉換為浮點數 (float)
        numeric_cols = ["open", "high", "low", "close", "volume", 
                       "quote_asset_volume", "taker_buy_base_asset_volume", 
                       "taker_buy_quote_asset_volume"]
        df[numeric_cols] = df[numeric_cols].astype(float)
        
        # 移除不需要的 'ignore' 欄位
        df = df.drop(columns=["ignore"])
        
        return df

    except requests.exceptions.RequestException as e:
        print(f"API 請求失敗: {e}")
        return None

# --- 主程式執行區 ---
if __name__ == "__main__":
    # 執行函數獲取數據
    df_klines = fetch_binance_futures_klines(symbol="BTCUSDT", interval="1m", limit=1500)
    
    if df_klines is not None:
        print(f"成功獲取 {len(df_klines)} 筆數據！")
        print("資料前 5 筆預覽：")
        print(df_klines.head())
        
        # 將數據儲存為 CSV 檔案
        filename = "BTCUSDT_futures_1m.csv"
        df_klines.to_csv(filename, index=False)
        print(f"\n數據已成功儲存至檔案：{filename}")

正在獲取 BTCUSDT 1m K線數據...
成功獲取 1500 筆數據！
資料前 5 筆預覽：
            open_time     open     high      low    close   volume  \
0 2026-04-20 16:59:00  75281.7  75300.0  75255.7  75300.0   34.261   
1 2026-04-20 17:00:00  75300.0  75332.0  75291.4  75320.8   52.373   
2 2026-04-20 17:01:00  75320.8  75344.6  75286.1  75305.6   37.810   
3 2026-04-20 17:02:00  75305.6  75449.1  75242.5  75377.1  230.476   
4 2026-04-20 17:03:00  75377.0  75377.0  75323.2  75350.3   88.720   

               close_time  quote_asset_volume  number_of_trades  \
0 2026-04-20 16:59:59.999        2.578821e+06              1334   
1 2026-04-20 17:00:59.999        3.944439e+06              2396   
2 2026-04-20 17:01:59.999        2.847949e+06              2325   
3 2026-04-20 17:02:59.999        1.736812e+07              7647   
4 2026-04-20 17:03:59.999        6.684824e+06              5425   

   taker_buy_base_asset_volume  taker_buy_quote_asset_volume  
0                       15.377                  1.157505e+06  


In [7]:
from data_loader import DataLoader
import os
import time
import logging
import traceback
from dotenv import load_dotenv
from binance.um_futures import UMFutures
load_dotenv()

key = os.getenv('BINANCE_API_KEY')
secret = os.getenv('BINANCE_SECRET_KEY')

client = UMFutures(key=key, secret=secret)
loader = DataLoader(client)
df_klines = loader.get_binance_klines(symbol="BTCUSDT", interval="1m", limit=15000)

if df_klines is not None:
    df_klines["open_time"] = pd.to_datetime(df_klines["open_time"], unit="ms")
    df_klines["close_time"] = pd.to_datetime(df_klines["close_time"], unit="ms")
    print(f"成功獲取 {len(df_klines)} 筆數據！")
    print("資料前 5 筆預覽：")
    print(df_klines.head())
    
    # 將數據儲存為 CSV 檔案
    filename = "BTCUSDT_futures_1m2.csv"
    df_klines.to_csv(filename, index=False)
    print(f"\n數據已成功儲存至檔案：{filename}")

幣安數據抓取失敗: (400, -1130, "Data sent for parameter 'limit' is not valid.", {'Content-Type': 'application/json', 'Content-Length': '68', 'Connection': 'keep-alive', 'Date': 'Tue, 21 Apr 2026 18:07:52 GMT', 'Server': 'Tengine', 'x-mbx-used-weight-1m': '10', 'x-response-time': '0ms', 'Access-Control-Allow-Origin': '*', 'Access-Control-Allow-Methods': 'GET, POST, PUT, DELETE, OPTIONS', 'X-Cache': 'Error from cloudfront', 'Via': '1.1 3c8be9297a43f45a4d7083ec511385e4.cloudfront.net (CloudFront)', 'X-Amz-Cf-Pop': 'TPE53-P2', 'X-Amz-Cf-Id': 'OcnQIeousKg0Ro941zvTeJBDN7c-5l7I0MV-CB5a3GzhIzyeiYSxuA=='}, None)


KeyError: 'open_time'

In [5]:
tw=loader.get_orderbook_depth(symbol="BTCUSDT")

In [6]:
tw

,timestamp,level,bid_price,bid_qty,ask_price,ask_qty
0,1776794455731,1,75541.8,9.726,75541.9,3.665
1,1776794455731,2,75541.7,0.004,75542.0,0.003
2,1776794455731,3,75541.6,0.002,75542.1,0.043
3,1776794455731,4,75541.5,0.002,75542.5,0.016
4,1776794455731,5,75541.2,0.001,75542.6,0.093


In [8]:
import os
import requests
import pandas as pd
from datetime import datetime, timedelta

def download_binance_daily_bookdepth(symbol="BTCUSDT", start_date="2024-04-01", end_date="2024-04-03", save_dir="binance_data"):
    """
    從 Binance Vision 下載指定日期範圍的 L2 Orderbook (bookDepth) 歷史數據壓縮檔
    """
    # 確保儲存目錄存在
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    # 將字串轉換為 datetime 物件，方便進行日期加減
    start = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")
    
    current_date = start
    downloaded_files = []

    print(f"準備下載 {symbol} 從 {start_date} 到 {end_date} 的 Orderbook 數據...\n")

    # 迴圈處理每一天
    while current_date <= end:
        date_str = current_date.strftime("%Y-%m-%d")
        file_name = f"{symbol}-bookDepth-{date_str}.zip"
        
        # Binance Vision 的固定下載網址結構 (U本位合約)
        url = f"https://data.binance.vision/data/futures/um/daily/bookDepth/{symbol}/{file_name}"
        save_path = os.path.join(save_dir, file_name)
        
        print(f"[{date_str}] 正在下載...")
        
        try:
            # 發送請求，stream=True 可以避免大檔案佔用過多記憶體
            response = requests.get(url, stream=True)
            
            # 檢查檔案是否存在 (如果日期太新，幣安可能還沒產生當天檔案，會回傳 404)
            if response.status_code == 200:
                with open(save_path, 'wb') as f:
                    for chunk in response.iter_content(chunk_size=8192):
                        f.write(chunk)
                print(f"  └─ 成功下載: {file_name}")
                downloaded_files.append(save_path)
            elif response.status_code == 404:
                print(f"  └─ 找不到檔案 (可能是官方尚未更新該日數據或日期錯誤): 404 Not Found")
            else:
                print(f"  └─ 下載失敗，狀態碼: {response.status_code}")
                
        except Exception as e:
            print(f"  └─ 發生錯誤: {e}")
            
        # 前進到下一天
        current_date += timedelta(days=1)
        
    print("\n下載任務完成！")
    return downloaded_files

# --- 主程式與 Pandas 讀取範例 ---
if __name__ == "__main__":
    # 1. 執行下載 (這裡示範下載 3 天的資料)
    symbol = "BTCUSDT"
    files = download_binance_daily_bookdepth(
        symbol=symbol, 
        start_date="2024-04-01", 
        end_date="2024-04-03", 
        save_dir="orderbook_data"
    )
    
    # 2. 示範：直接用 Pandas 讀取剛剛下載的 ZIP 檔 (不需手動解壓縮)
    if files:
        sample_file = files[0]
        print(f"\n嘗試讀取剛下載的檔案: {sample_file}")
        
        # Pandas 支援直接讀取 .zip 裡面的 csv，非常方便！
        # 注意：歷史數據檔案很大，這裡用 nrows=5 只讀取前 5 筆做測試
        df_depth = pd.read_csv(sample_file, compression='zip', nrows=5)
        
        print("資料前 5 筆預覽：")
        print(df_depth)

準備下載 BTCUSDT 從 2024-04-01 到 2024-04-03 的 Orderbook 數據...

[2024-04-01] 正在下載...
  └─ 成功下載: BTCUSDT-bookDepth-2024-04-01.zip
[2024-04-02] 正在下載...
  └─ 成功下載: BTCUSDT-bookDepth-2024-04-02.zip
[2024-04-03] 正在下載...
  └─ 成功下載: BTCUSDT-bookDepth-2024-04-03.zip

下載任務完成！

嘗試讀取剛下載的檔案: orderbook_data\BTCUSDT-bookDepth-2024-04-01.zip
資料前 5 筆預覽：
             timestamp  percentage     depth      notional
0  2024-04-01 00:00:09          -5  7979.613  5.554046e+08
1  2024-04-01 00:00:09          -4  6210.110  4.347691e+08
2  2024-04-01 00:00:09          -3  4679.587  3.293200e+08
3  2024-04-01 00:00:09          -2  3620.459  2.555657e+08
4  2024-04-01 00:00:09          -1  1406.623  9.991825e+07


In [9]:
pip install pybit

Note: you may need to restart the kernel to use updated packages.
